# 10 — MPC目的関数と重み

QとRを「ゲイン」として暗記せず、予測誤差と入力使用量の交換条件として読みます。

**前提**: `09_contact_and_friction_constraints.ipynb`

> 読み方: 「直感 → 数式 → 上流コード → 小実験 → 解釈」の順です。
> `実装事実` と書いた箇所は現行 `external/Quadruped-PyMPC` のコード、
> `学習用モデル` は理解のために単純化した再実装です。

In [1]:
from pathlib import Path
import os, sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebook_pympc":
    ROOT = ROOT.parent
PYMPC_ROOT = ROOT / "external" / "Quadruped-PyMPC"
assert PYMPC_ROOT.exists(), f"Quadruped-PyMPC が見つかりません: {PYMPC_ROOT}"
if str(PYMPC_ROOT) not in sys.path:
    sys.path.insert(0, str(PYMPC_ROOT))

os.environ.setdefault("ACADOS_SOURCE_DIR", str(PYMPC_ROOT / "quadruped_pympc" / "acados"))
os.environ.setdefault("MUJOCO_GL", "egl")
print("workspace :", ROOT)
print("PyMPC root:", PYMPC_ROOT)

workspace : /home/takuya/work/mpc_dog
PyMPC root: /home/takuya/work/mpc_dog/external/Quadruped-PyMPC


## 有限時間最適化

\[
\min_{\{x_k,u_k\}}
\sum_{k=0}^{N-1}
  (x_k-x_k^{ref})^TQ(x_k-x_k^{ref})
  +(u_k-u_k^{ref})^TR(u_k-u_k^{ref})
+(x_N-x_N^{ref})^TQ_N(x_N-x_N^{ref})
\]

subject to SRBD dynamics、初期状態、摩擦・接触制約。

\(Q\) を増やすとその状態誤差を高価にし、\(R\) を増やすと入力を使うことを高価にします。
単位が違うため、重みの数値の大小だけを横比較してはいけません。

In [2]:
import numpy as np

# set_weight()が作る対角Qの3成分を、典型誤差で切り出して読む。
# stage cost: l_k = (x-x_ref).T @ Q @ (x-x_ref) + (u-u_ref).T @ R @ (u-u_ref)
errors = {
    "z [m]": 0.03,                  # e_z
    "vx [m/s]": 0.20,              # e_vx
    "roll [rad]": np.deg2rad(5),   # e_roll。degreeのまま入れない
}
weights = {"z [m]": 1500, "vx [m/s]": 100, "roll [rad]": 500}
for name, error in errors.items():
    # 対角Qなので、この状態の寄与は単純にq_i*e_i^2。
    contribution = weights[name] * error**2
    print(f"{name:12s}: error={error:.4f}, q_i*e_i^2={contribution:.3f}")

# 数値が最大の重みではなく、実際の誤差を掛けた寄与を比較する。
assert weights["z [m]"] > weights["roll [rad]"]

z [m]       : error=0.0300, contribution=1.350
vx [m/s]    : error=0.2000, contribution=4.000
roll [rad]  : error=0.0873, contribution=3.808


## 現行コードの読み方

`centroidal_nmpc_nominal.py::set_weight(nx,nu)` が対角Q/Rを作り、
`scipy.linalg.block_diag(Q,R)` がstage costのWになります。
`Vx` と `Vu` は `(x,u)` をcost出力 `y` へ並べる選択行列です。

変更前に、対象状態のindex・単位・典型誤差を確認し、
`weight × error²` の寄与を比較します。

In [3]:
# 正規化した学習用比較: 同じ物理誤差でもscaleで意味が変わる
scales = {"z": 0.05, "vx": 0.5, "roll": np.deg2rad(10)}
for key, scale in scales.items():
    print(key, "unit normalized weight =", 1/scale**2)

z unit normalized weight = 399.99999999999994
vx unit normalized weight = 4.0
roll unit normalized weight = 32.828063500117445


## 章末チェック

出力を眺めるだけでなく、次を自分の言葉で答えてください。

1. この章の入力・出力の shape、単位、座標系は何か。
2. 変更可能な量と、他の章から渡される量は何か。
3. パラメータを2倍にしたとき、どのグラフがどちらへ変化するか。
4. 現行実装の事実と、学習用の近似を区別できるか。